In [1]:
import pennylane as qml
import numpy as np

# ==========================================
# PARTE 1: I DATI E I REGISTRI/FILI
# ==========================================
# 1. Partiamo da una VERA matrice diagonale 4x4 (L'input)
# X_matrice = np.array([
#     [ 0.5,   0,   0,   0],
#     [  0, -0.2,   0,   0],
#     [  0,   0, 0.8,   0],
#     [  0,   0,   0,   0.1]
# ])

X_matrice = np.array([
    [ 0.7,   0,   0,   0],
    [  0, -4,   0,   0],
    [  0,   0, 8,   0],
    [  0,   0,   0,   0.3]
])

# X_matrice = np.array([
#     [ -720,   0,   0,   0],
#     [  0, 56,   0,   0],
#     [  0,   0, 200,   0],
#     [  0,   0,   0,   170]
# ])

# X_matrice = np.array([
#     [ -720,   0],
#     [  0, 56]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0],
#     [  0, 56,   0],
#     [  0,   0, 200],
# ])

print("1. La matrice X di partenza è:")
print(X_matrice)


# 2. Estraiamo la diagonale classica
x_classico_grezzo = np.diag(X_matrice)
N_originale = len(x_classico_grezzo)


# 3. Calcolo qubit e Padding
# Calcoliamo n: il numero di qubit necessari (logaritmo in base 2 arrotondato per eccesso)
# Es: N=2 -> n=1. N=3 -> n=2. N=4 -> n=2.
n = int(np.ceil(np.log2(N_originale)))

# Calcoliamo quanti slot fisici avrà la memoria quantistica (la potenza di 2 più vicina)
N_pad = 2**n

# Facciamo il padding: aggiungiamo zeri alla fine se N_originale è minore di N_pad
# (Fondamentale per le matrici 3x3, 5x5, ecc.)
x_classico = np.pad(x_classico_grezzo, (0, N_pad - N_originale))


# 4. Normalizzazione del vettore (obbligatoria per le ampiezze, sennò il computer quantistico non può leggere i dati)
norma_originale = np.linalg.norm(x_classico)
x = x_classico / norma_originale 


# 5. Assegniamo i fili. n=2 (perché abbiamo 4 dati, 2^2=4)
# Servono n fili per address, n per data, 1 per l'ancilla B, 1 per l'ancilla lcu.
# Generiamo le liste dei fili in base alla 'n' calcolata!
ancilla_lcu = [0]                                    # Il nuovo qubit (+1) in alto
qubits_address = list(range(1, n+1))                   # Es: se n=1 -> [0]
qubits_dati = list(range(n+1, 2*n +1))                    # Es: se n=1 -> [1]
ancilla_b = [2*n +1]                                    # Es: se n=1 -> [2]

# Prepariamo un raggruppamento che ci servirà per S_0
qubits_da_B = qubits_dati + ancilla_b

# Creiamo una lista unica con tutti i 5 fili nell'ordine corretto
tutti_i_fili = ancilla_lcu + qubits_address + qubits_dati + ancilla_b



# ==========================================
# PARTE 2: ORACOLO E RIFLESSIONE
# ==========================================

def U_x(wires):
    """
    L'Oracolo: Inietta il vettore x nelle ampiezze dei qubit bersaglio.
    """
     #qml.AmplitudeEmbedding(features=x, wires=wires, normalize=True)
    qml.MottonenStatePreparation(x, wires=wires)

def S_0():
    """
    La Riflessione: La matrice diagonale con un -1 in alto a sinistra.
    Inverte il segno SOLO se i qubit 'dati' e 'B' sono tutti a zero.
    Non tocca gli 'address' per non rovinare le coordinate!
    """
    stato_zero = np.zeros(len(qubits_da_B), dtype=int)
    qml.FlipSign(stato_zero, wires=qubits_da_B)



# ==========================================
# PARTE 3: W (dalla Figura 1)
# ==========================================

def W():
    """
    Allinea i dati x_k con le coordinate corrette della matrice diagonale.
    """
    # 1. U normale solo sui dati
    U_x(wires=qubits_dati)
    
    # 2. Hadamard sull'ancilla B
    qml.Hadamard(wires=ancilla_b)
    
    # 3. U inverso sui dati, controllato da B
    # qml.adjoint ribalta l'oracolo, qml.ctrl lo subordina a B
    qml.ctrl(qml.adjoint(U_x), control=ancilla_b)(wires=qubits_dati)
    
    # 4. Le porte Toffoli incrociate (da address a dati, controllate da B)
    # Servono a "saldare" il dato corretto all'indirizzo corretto
    for i in range(len(qubits_address)):
        # Toffoli prende: [controllo_1, controllo_2, bersaglio]
        qml.Toffoli(wires=[ancilla_b[0], qubits_address[i], qubits_dati[i]])
        
    # 5. Hadamard finale sull'ancilla B
    qml.Hadamard(wires=ancilla_b)



# ==========================================
# PARTE 4: G
# ==========================================

def G():
    # Costruisce l'operatore G = W S_0 W^\dagger Z_B.
    # Le operazioni si scrivono nell'ordine esatto in cui toccano i qubit.
    
    # 1. Z_B (Riflessione sull'ancilla)
    qml.PauliZ(wires=ancilla_b)
    
    # 2. W^\dagger (Smontaggio: PennyLane inverte tutto W in automatico)
    qml.adjoint(W)()
    
    # 3. S_0 (Riflessione sull'origine)
    S_0()
    
    # 4. W (Rimontaggio)
    W()



# ==========================================
# PARTE 5: Gtilde
# ==========================================

def G_tilde():
    """Implementa -1/2 (G + G_dagger) usando le porte esatte del paper"""
    
    # 1. Hadamard iniziale sull'ancilla LCU
    qml.Hadamard(wires=ancilla_lcu)
    
    # 2. Sandwich PauliX per il controllo su 0 (Pallino bianco)
    qml.PauliX(wires=ancilla_lcu)
    qml.ctrl(G, control=ancilla_lcu)()  # G controllato standard
    qml.PauliX(wires=ancilla_lcu)
    
    # 3. Controllo su 1 per G_dagger (Pallino nero)
    qml.ctrl(qml.adjoint(G), control=ancilla_lcu)()
    
    # 4. Hadamard finale per ricongiungere gli stati
    qml.Hadamard(wires=ancilla_lcu)
    
    # 5. La rotazione Rz(2pi) per ottenere il segno MENO globale
    qml.RZ(2 * np.pi, wires=ancilla_lcu)

    
    
# ==========================================
# PARTE 6: QSVT (Il Programma Non Lineare)
# ==========================================

# 1. Definiamo il nostro polinomio di grado 3: P(x) = ax^3 + bx^2 + cx + d
a, b, c, d_coeff = 0.5, -0.2, 0.1, 0.05 

print(f"\n--- IMPOSTAZIONE QSVT ---")
print(f"Vogliamo applicare il polinomio: P(x) = {a}x^3 + {b}x^2 + {c}x + {d_coeff}")

# 2. La Traduzione in Angoli di Fase (Phase Angles)
# In un ambiente di produzione reale, qui chiameresti un "angle solver" classico.
# Qui inseriamo 4 angoli fittizi (in radianti) per costruire fisicamente il circuito.
angoli_fase_qsvt = np.array([0.15, -0.42, 0.88, -0.15]) 
print(f"Angoli di fase pre-calcolati per il circuito: {angoli_fase_qsvt}")

# 3. PennyLane vuole un "Operatore" formale per la QSVT. Quindi calcoliamo la matrice 
# esatta del nostro G_tilde e la convertiamo in un operatore unitario nativo.
matrice_G_tilde = qml.matrix(G_tilde, wire_order=tutti_i_fili)()
operatore_G_tilde = qml.QubitUnitary(matrice_G_tilde, wires=tutti_i_fili)
    
# 4. Definiamo i "fili proiettore". 
# A noi serve langolino in alto a sinistra che corrisponde allo stato in cui 
# l'ancilla LCU, l'ancilla B e i qubit dei Dati sono tutti a ZERO. 
# La QSVT ha bisogno di saperlo per agire sul posto giusto!

# Definiamo i fili e lo stato (tutti zeri) dell'angolino in alto a sinistra
fili_proiettore = ancilla_lcu + qubits_dati + ancilla_b
stato_zero = np.zeros(len(fili_proiettore), dtype=int)

# Creiamo una lista di Phase-Shift Controllers (uno per ogni angolo).
# 'dim=1' significa che la fase viene applicata ESATTAMENTE e SOLO allo stato 
# in cui i fili del proiettore sono a ZERO (il nostro angolino in alto a sinistra!).
lista_proiettori = [qml.PCPhase(angle, dim=1, wires=fili_proiettore) for angle in angoli_fase_qsvt]


def circuito_qsvt():
    """
    Applica la QSVT al block-encoding G_tilde.
    Prende: 1. Operatore Base | 2. Lista di [ProiettoreIn, ProiettoreOut] | 3. Angoli
    """
    qml.QSVT(operatore_G_tilde, lista_proiettori)
    
# 3. Test di Compilazione del Circuito
dev_test = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_test)
def test_qsvt_build():
    # Prepariamo lo stato iniziale: Hadamard sugli Address per creare 
    # la sovrapposizione uniforme di tutti gli input.
    for wire in qubits_address:
        qml.Hadamard(wires=wire)
        
    # Applichiamo la scatola QSVT che incastra G_tilde + Polinomio
    circuito_qsvt()
    
    # Ritorniamo il vettore di stato finale
    return qml.state()

print("\n--- COMPILAZIONE CIRCUITO ---")
try:
    stato_finale = test_qsvt_build()
    print("-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!")
    print(f"-> Il vettore di stato risultante ha {len(stato_finale)} ampiezze complesse.")
except Exception as e:
    print(f"-> Errore nella compilazione QSVT: {e}")
    
    

# ==========================================
# PARTE 7: CIRCUITO FINALE ED ESTRAZIONE (Fig. 4)
# ==========================================

print("\n--- ESECUZIONE CIRCUITO NTCA FINALE ---")

print("\n--- COSTRUZIONE DEL GATE P_CORSIVO E DELLO STATO FINALE ---")

# 1. Definiamo formalmente il GATE P_corsivo
def P_corsivo():
    """Il Gate P_corsivo: la trasformazione non lineare sui valori singolari."""
    # Usiamo 'operatore_G_tilde' e 'lista_proiettori' già definiti nella Parte 6
    qml.QSVT(operatore_G_tilde, lista_proiettori)

# 2. Costruiamo lo STATO quantistico seguendo il disegno "GUESS"
dev_finale = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_finale)
def circuito_stato_finale():
    """
    Traduzione esatta del disegno 'GUESS' da sinistra a destra:
    H su address -> W -> P_corsivo -> W_dagger
    """
    
    # a) H sul registro 'add' (Inizializzazione della sovrapposizione)
    for wire in qubits_address:
        qml.Hadamard(wires=wire)
        
    # b) Gate W (Allinea i dati con gli indirizzi)
    W()
    
    # c) Gate P_corsivo (Applica il polinomio QSVT)
    P_corsivo()
    
    # d) Gate W_dagger (Smonta l'allineamento per isolare i risultati puliti)
    qml.adjoint(W)()
    
    # Restituisce il vettore di stato gigante da cui pescheremo i risultati
    return qml.state()

# 3. Esecuzione ed Estrazione
print("-> Esecuzione del circuito 'GUESS' in corso...")
stato_ntca = circuito_stato_finale()
print("   Circuito eseguito! Vettore di stato ottenuto.\n")

print("--- RISULTATI ESTRATTI (Post-selezione) ---")
risultati_quantistici = np.zeros(N_pad, dtype=complex)

# Il vettore ha 64 elementi. Cerchiamo la fetta in cui b_anc, data e B sono a 0.
for i, ampiezza in enumerate(stato_ntca):
    # Convertiamo l'indice in stringa binaria (es: '010110')
    binario = format(i, f'0{len(tutti_i_fili)}b')
    
    lcu_str = binario[0]                   # 'b_anc' nel disegno
    addr_str = binario[1:1+n]              # 'add' nel disegno
    dati_e_b_str = binario[1+n:]           # 'data' e 'B' nel disegno
    
    # La regola: prendi solo se 'b_anc' è 0 E tutti i 'data' e 'B' sono 0
    if lcu_str == '0' and all(bit == '0' for bit in dati_e_b_str):
        k = int(addr_str, 2) # Converte l'indirizzo da binario a decimale
        # Moltiplichiamo per sqrt(N_pad) per compensare le porte H iniziali
        risultati_quantistici[k] = ampiezza * np.sqrt(N_pad)

# Stampiamo i valori finali trovati negli address |k>
for k in range(N_pad):
    valore = risultati_quantistici[k]
    # Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
    valore_pulito = np.round(valore.real, 4) + 1j * np.round(valore.imag, 4)
    print(f"Indirizzo |{k}> : {valore_pulito}")

1. La matrice X di partenza è:
[[ 0.7  0.   0.   0. ]
 [ 0.  -4.   0.   0. ]
 [ 0.   0.   8.   0. ]
 [ 0.   0.   0.   0.3]]

--- IMPOSTAZIONE QSVT ---
Vogliamo applicare il polinomio: P(x) = 0.5x^3 + -0.2x^2 + 0.1x + 0.05
Angoli di fase pre-calcolati per il circuito: [ 0.15 -0.42  0.88 -0.15]

--- COMPILAZIONE CIRCUITO ---
-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!
-> Il vettore di stato risultante ha 64 ampiezze complesse.

--- ESECUZIONE CIRCUITO NTCA FINALE ---

--- COSTRUZIONE DEL GATE P_CORSIVO E DELLO STATO FINALE ---
-> Esecuzione del circuito 'GUESS' in corso...
   Circuito eseguito! Vettore di stato ottenuto.

--- RISULTATI ESTRATTI (Post-selezione) ---
Indirizzo |0> : (-0.0481-0.0145j)
Indirizzo |1> : (-0.3994+0.1972j)
Indirizzo |2> : (0.7933-0.3944j)
Indirizzo |3> : (0.0288-0.0148j)
